In [2]:
import numpy as np
from scipy.spatial.distance import cdist

# =====================================================================
# CHỨC NĂNG 1: CÁC HÀM CỦA THUẬT TOÁN K-MEANS (TỪ BÀI 1)
# =====================================================================

def kmeans_init_centers(X, n_cluster):
    # Khởi tạo ngẫu nhiên n_cluster tâm cụm không trùng lặp từ bộ dữ liệu X
    return X[np.random.choice(X.shape[0], n_cluster, replace=False)]

def kmeans_predict_labels(X, centers):
    # Tính ma trận khoảng cách Euclid giữa toàn bộ điểm dữ liệu X và các tâm cụm centers
    D = cdist(X, centers)
    # Trả về chỉ số (index) của tâm cụm gần nhất cho từng điểm dữ liệu
    return np.argmin(D, axis=1)

def kmeans_update_centers(X, labels, n_cluster):
    # Cập nhật lại vị trí các tâm cụm bằng cách lấy trung bình cộng tọa độ của các điểm thuộc cụm đó
    centers = np.zeros((n_cluster, X.shape[1]))
    for k in range(n_cluster):
        Xk = X[labels == k, :]
        if len(Xk) > 0:
            centers[k, :] = np.mean(Xk, axis=0)
    return centers

def kmeans_has_converged(centers, new_centers):
    # Trả về True nếu tập hợp các tâm cụm mới trùng khớp hoàn toàn với tập hợp tâm cụm cũ
    return (set([tuple(a) for a in centers]) == set([tuple(a) for a in new_centers]))

def fit_kmeans(X, n_cluster, max_iters=100):
    centers = kmeans_init_centers(X, n_cluster)
    for i in range(max_iters):
        labels = kmeans_predict_labels(X, centers)
        new_centers = kmeans_update_centers(X, labels, n_cluster)
        if kmeans_has_converged(centers, new_centers):
            break
        centers = new_centers
    return centers, labels


# =====================================================================
# CHỨC NĂNG 2: CÁC HÀM CỦA THUẬT TOÁN K-NN (TỪ BÀI 2)
# =====================================================================

def KNN(X_train, X_test, y_train, k):
    num_test = X_test.shape[0]
    num_train = X_train.shape[0]
    y_pred = np.zeros((num_test, num_train))

    # Tính ma trận khoảng cách giữa các điểm Test và dữ liệu Train
    for i in range(num_test):
        for j in range(num_train):
            y_pred[i, j] = np.sqrt(np.sum(np.power(X_test[i, :] - X_train[j, :], 2)))

    results = []
    # Sắp xếp khoảng cách tăng dần và lấy k láng giềng gần nhất
    for i in range(len(y_pred)):
        zipped = zip(y_pred[i, :], y_train)
        res = sorted(zipped, key=lambda x: x[0])
        results_topk = res[:k]

        # Đếm số lượng phiếu bầu của mỗi lớp bằng từ điển (Hòa phiếu lấy lớp xuất hiện trước)
        classes = {}
        for _, j in results_topk:
            j = int(j)
            if j not in classes:
                classes[j] = 1
            else:
                classes[j] = classes[j] + 1

        results.append(max(classes, key=classes.get))
    return np.array(results)


# =====================================================================
# KHỐI THỬ NGHIỆM VÀ CHẠY DEMO ỨNG DỤNG BÀI 3
# =====================================================================
if __name__ == "__main__":
    print("=========================================================")
    print("      CHƯƠNG TRÌNH KIỂM THỬ ỨNG DỤNG SONG SONG (BÀI 3)   ")
    print("=========================================================\n")

    # Khởi tạo một tập dữ liệu thử nghiệm chung gồm 6 điểm được phân bố rõ rệt thành 2 nhóm
    # Nhóm 1: quanh khu vực (1, 1) | Nhóm 2: quanh khu vực (8, 8)
    X_data = np.array([
        [1.0, 1.0],
        [1.5, 1.2],
        [2.0, 1.0],
        [8.0, 8.0],
        [8.5, 9.0],
        [9.0, 8.5]
    ])

    # Nhãn phân loại thực tế của 6 điểm trên (3 điểm đầu lớp 0, 3 điểm sau lớp 1)
    y_data = np.array([0, 0, 0, 1, 1, 1])

    # -----------------------------------------------------------------
    # THỬ NGHIỆM 1: CHẠY THUẬT TOÁN PHÂN CỤM K-MEANS
    # -----------------------------------------------------------------
    print("[THỬ NGHIỆM 1] Thực thi thuật toán K-Means (Gom cụm tự động):")
    km_centers, km_labels = fit_kmeans(X_data, n_cluster=2)

    print(" -> Tọa độ các tâm cụm tối ưu sau khi hội tụ:\n", km_centers)
    print(" -> Nhãn phân cụm tự động gán cho 6 điểm dữ liệu:", km_labels)
    print("-" * 57)

    # -----------------------------------------------------------------
    # THỬ NGHIỆM 2: CHẠY THUẬT TOÁN PHÂN LOẠI K-NN
    # -----------------------------------------------------------------
    print("\n[THỬ NGHIỆM 2] Thực thi thuật toán K-NN (Dự đoán điểm mới):")

    # Đặt 2 điểm dữ liệu truy vấn mới cần kiểm tra nhãn lớp
    # Điểm truy vấn thứ nhất gần Nhóm 1, điểm truy vấn thứ hai gần Nhóm 2
    X_query = np.array([
        [1.2, 1.1],
        [8.2, 8.3]
    ])

    # Gọi hàm KNN dự đoán nhãn với tham số láng giềng k = 3
    pred_labels = KNN(X_data, X_query, y_data, k=3)

    print(" -> Tọa độ các điểm kiểm thử mới cần dán nhãn:\n", X_query)
    print(" -> Kết quả dự đoán nhãn lớp tương ứng từ mô hình K-NN:", pred_labels)
    print("=========================================================")

      CHƯƠNG TRÌNH KIỂM THỬ ỨNG DỤNG SONG SONG (BÀI 3)   

[THỬ NGHIỆM 1] Thực thi thuật toán K-Means (Gom cụm tự động):
 -> Tọa độ các tâm cụm tối ưu sau khi hội tụ:
 [[1.5        1.06666667]
 [8.5        8.5       ]]
 -> Nhãn phân cụm tự động gán cho 6 điểm dữ liệu: [0 0 0 1 1 1]
---------------------------------------------------------

[THỬ NGHIỆM 2] Thực thi thuật toán K-NN (Dự đoán điểm mới):
 -> Tọa độ các điểm kiểm thử mới cần dán nhãn:
 [[1.2 1.1]
 [8.2 8.3]]
 -> Kết quả dự đoán nhãn lớp tương ứng từ mô hình K-NN: [0 1]
